### ILI MODEL

In [ ]:
import math
import torch
import torch.nn.functional as F


def _to_float_tensor(x):
    if torch.is_tensor(x):
        if getattr(x, "is_sparse", False) or getattr(x, "is_sparse_csr", False) or getattr(x, "is_sparse_bsr", False):
            x = x.to_dense()
        return x.float()
    if hasattr(x, "toarray"):
        x = x.toarray()
    elif hasattr(x, "A"):
        x = x.A
    return torch.tensor(x, dtype=torch.float32)


def _to_long_tensor(x):
    if torch.is_tensor(x):
        if getattr(x, "is_sparse", False) or getattr(x, "is_sparse_csr", False) or getattr(x, "is_sparse_bsr", False):
            x = x.to_dense()
        return x.long()
    if hasattr(x, "toarray"):
        x = x.toarray()
    elif hasattr(x, "A"):
        x = x.A
    return torch.tensor(x, dtype=torch.long)


def _build_adjacency(graph, n_nodes, device):
    if graph is None:
        A = torch.eye(n_nodes, device=device, dtype=torch.float32)
    else:
        if torch.is_tensor(graph):
            g = graph.detach()
            if getattr(g, "layout", torch.strided) != torch.strided:
                g = g.to_dense()
            g = g.float()

            if g.ndim == 2 and g.shape[0] == n_nodes and g.shape[1] == n_nodes:
                A = g.to(device)
                A = 0.5 * (A + A.t())
            elif g.ndim == 2 and (g.shape[0] == 2 or g.shape[1] == 2):
                edge_index = g.long()
                if edge_index.shape[0] != 2:
                    edge_index = edge_index.t().contiguous()
                src = edge_index[0].clamp(0, n_nodes - 1).to(device)
                dst = edge_index[1].clamp(0, n_nodes - 1).to(device)
                A = torch.zeros((n_nodes, n_nodes), device=device, dtype=torch.float32)
                vals = torch.ones(src.numel(), device=device, dtype=torch.float32)
                A.index_put_((src, dst), vals, accumulate=True)
                A.index_put_((dst, src), vals, accumulate=True)
            else:
                raise ValueError("Unsupported graph tensor shape.")
        elif hasattr(graph, "tocoo"):
            coo = graph.tocoo()
            row = torch.as_tensor(coo.row, device=device, dtype=torch.long).clamp(0, n_nodes - 1)
            col = torch.as_tensor(coo.col, device=device, dtype=torch.long).clamp(0, n_nodes - 1)
            data = torch.as_tensor(coo.data, device=device, dtype=torch.float32)
            A = torch.zeros((n_nodes, n_nodes), device=device, dtype=torch.float32)
            A.index_put_((row, col), data, accumulate=True)
            A.index_put_((col, row), data, accumulate=True)
        elif isinstance(graph, (tuple, list)) and len(graph) in (2, 3):
            edge_index = _to_long_tensor(graph[0]).to(device)
            if edge_index.ndim != 2:
                raise ValueError("Edge index input must be 2D.")
            if edge_index.shape[0] != 2:
                edge_index = edge_index.t().contiguous()

            src = edge_index[0].clamp(0, n_nodes - 1)
            dst = edge_index[1].clamp(0, n_nodes - 1)
            A = torch.zeros((n_nodes, n_nodes), device=device, dtype=torch.float32)

            if len(graph) >= 2 and graph[1] is not None:
                vals = _to_float_tensor(graph[1]).to(device).reshape(-1)
                if vals.numel() != src.numel():
                    vals = torch.ones(src.numel(), device=device, dtype=torch.float32)
            else:
                vals = torch.ones(src.numel(), device=device, dtype=torch.float32)

            A.index_put_((src, dst), vals, accumulate=True)
            A.index_put_((dst, src), vals, accumulate=True)
        else:
            A = _to_float_tensor(graph).to(device)
            if A.ndim != 2 or A.shape[0] != n_nodes or A.shape[1] != n_nodes:
                raise ValueError("graph adjacency must match the node dimension of input.")
            A = 0.5 * (A + A.t())

    A = torch.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0).clamp_min(0.0)
    A = A + torch.eye(n_nodes, device=device, dtype=torch.float32)

    deg = A.sum(dim=1).clamp_min(1e-6)
    deg_inv_sqrt = deg.rsqrt()
    A = deg_inv_sqrt[:, None] * A * deg_inv_sqrt[None, :]
    return A, deg


def simple_tabular_st(
    train_input,
    train_target,
    x_eval=None,
    train_states=None,
    eval_states=None,
    graph=None,
    tid_sizes=None,
):
    graph=None
    x_eval = train_input if x_eval is None else x_eval
    eval_states = train_states if eval_states is None else eval_states
    tid_sizes = {"woy": 53} if tid_sizes is None else tid_sizes

    compute_device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    if not isinstance(compute_device, torch.device):
        compute_device = torch.device(compute_device)

    def _normalize_series_4d_to_3d(x):
        x = _to_float_tensor(x)
        if x.ndim == 4:
            if x.shape[-1] == 1:
                x = x[..., 0]
            else:
                x = x.mean(dim=-1)
        return x

    def _orient_series_x(x, expected_nodes=None):
        x = _normalize_series_4d_to_3d(x)
        if x.ndim == 1:
            x = x.unsqueeze(0).unsqueeze(-1)
        elif x.ndim == 2:
            if expected_nodes is not None and x.shape[1] == expected_nodes:
                x = x.unsqueeze(1)
            else:
                x = x.unsqueeze(-1)
        if x.ndim != 3:
            raise ValueError("train_input/x_eval must have shape [B, L, N] or [B, N, L] or [B, L, N, C].")

        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        if expected_nodes is not None:
            if x.shape[-1] == expected_nodes:
                return x.contiguous()
            if x.shape[1] == expected_nodes:
                return x.transpose(1, 2).contiguous()

        return x.contiguous()

    def _orient_series_y(y, expected_nodes=None):
        y = _normalize_series_4d_to_3d(y)
        if y.ndim == 1:
            y = y.unsqueeze(0).unsqueeze(-1)
        elif y.ndim == 2:
            y = y.unsqueeze(-1)
        if y.ndim != 3:
            raise ValueError("train_target must have shape [B, H, N], [B, N, H], [B, H, N, C], or [B, H].")

        y = torch.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

        if expected_nodes is not None:
            if y.shape[-1] == expected_nodes:
                return y.contiguous()
            if y.shape[1] == expected_nodes:
                return y.transpose(1, 2).contiguous()

        return y.contiguous()

    if train_states is None or eval_states is None:
        raise ValueError("train_states and eval_states are required for spatiotemporal forecasting.")

    X_train_raw = _normalize_series_4d_to_3d(train_input)
    Y_train_raw = _normalize_series_4d_to_3d(train_target)

    if Y_train_raw.ndim == 2:
        Y_train_raw = Y_train_raw.unsqueeze(-1)

    if Y_train_raw.ndim != 3:
        raise ValueError("train_target must be 3D after preprocessing.")

    target_nodes = Y_train_raw.shape[-1] if Y_train_raw.shape[-1] > 1 else None
    X_train = _orient_series_x(X_train_raw, expected_nodes=target_nodes)
    Y_train = _orient_series_y(Y_train_raw, expected_nodes=X_train.shape[-1])
    X_eval = _orient_series_x(x_eval, expected_nodes=X_train.shape[-1])

    if X_train.ndim != 3 or X_eval.ndim != 3 or Y_train.ndim != 3:
        raise ValueError("All series inputs must be 3D after preprocessing.")

    if Y_train.shape[-1] != X_train.shape[-1]:
        if Y_train.shape[1] == X_train.shape[-1]:
            Y_train = Y_train.transpose(1, 2).contiguous()
        else:
            raise ValueError("train_target must match the node dimension of train_input for forecasting.")

    n_nodes = X_train.shape[-1]
    if X_eval.shape[-1] != n_nodes and X_eval.shape[1] == n_nodes:
        X_eval = X_eval.transpose(1, 2).contiguous()

    X_train = torch.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    Y_train = torch.nan_to_num(Y_train, nan=0.0, posinf=0.0, neginf=0.0)
    X_eval = torch.nan_to_num(X_eval, nan=0.0, posinf=0.0, neginf=0.0)

    S_train = _to_long_tensor(train_states).to(compute_device, non_blocking=True)
    S_eval = _to_long_tensor(eval_states).to(compute_device, non_blocking=True)

    X_train = X_train.to(compute_device, non_blocking=True)
    Y_train = Y_train.to(compute_device, non_blocking=True)
    X_eval = X_eval.to(compute_device, non_blocking=True)

    B, lookback, n_nodes = X_train.shape
    horizon = Y_train.shape[1]
    lookback = max(int(lookback), 1)

    X_train = torch.log1p(X_train.clamp_min(0.0))
    Y_train = torch.log1p(Y_train.clamp_min(0.0))
    X_eval = torch.log1p(X_eval.clamp_min(0.0))

    A, raw_deg = _build_adjacency(graph, n_nodes, compute_device)
    A2 = A @ A
    A3 = A2 @ A

    if B <= 1:
        n_val = 0
    else:
        n_val = max(1, B // 5)
        n_val = min(n_val, B - 1)

    if n_val > 0:
        X_tr = X_train[:-n_val]
        Y_tr = Y_train[:-n_val]
        S_tr = S_train[:-n_val]
        X_va = X_train[-n_val:]
        Y_va = Y_train[-n_val:]
        S_va = S_train[-n_val:]
    else:
        X_tr = X_train
        Y_tr = Y_train
        S_tr = S_train
        X_va = X_train
        Y_va = Y_train
        S_va = S_train

    train_scale_source = torch.cat([X_tr.reshape(-1, n_nodes), Y_tr.reshape(-1, n_nodes)], dim=0)
    node_mean = train_scale_source.mean(dim=0)
    node_std = train_scale_source.std(dim=0, unbiased=False).clamp(min=1e-6)

    X_tr = (X_tr - node_mean[None, None, :]) / node_std[None, None, :]
    Y_tr = (Y_tr - node_mean[None, None, :]) / node_std[None, None, :]
    X_va = (X_va - node_mean[None, None, :]) / node_std[None, None, :]
    Y_va = (Y_va - node_mean[None, None, :]) / node_std[None, None, :]
    X_eval_std = (X_eval - node_mean[None, None, :]) / node_std[None, None, :]

    woy_size = max(int(tid_sizes.get("woy", 53)), 1)
    woy_period = float(woy_size)

    def make_time_features(states):
        if not torch.is_tensor(states):
            states = _to_long_tensor(states)
        states = states.to(compute_device)

        if states.ndim == 1:
            primary = states[:, None].repeat(1, horizon)
        elif states.ndim == 2:
            primary = states
        elif states.ndim >= 3:
            primary = states[..., 0]
        else:
            raise ValueError("states must have shape [B], [B, H], or [B, H, K].")

        if primary.shape[1] < horizon:
            pad = primary[:, -1:].repeat(1, horizon - primary.shape[1])
            primary = torch.cat([primary, pad], dim=1)
        elif primary.shape[1] > horizon:
            primary = primary[:, :horizon]

        primary = primary.float()

        woy = torch.remainder(primary, woy_period) / woy_period
        hpos = torch.linspace(0.0, 1.0, horizon, device=states.device, dtype=torch.float32)
        hpos = hpos.unsqueeze(0).expand(states.shape[0], -1)

        twopi = 2.0 * math.pi
        return torch.stack(
            [
                torch.sin(twopi * woy),
                torch.cos(twopi * woy),
                torch.sin(2.0 * twopi * woy),
                torch.cos(2.0 * twopi * woy),
                torch.sin(twopi * hpos),
                torch.cos(twopi * hpos),
                hpos,
                hpos ** 2,
                hpos ** 3,
            ],
            dim=-1,
        )

    T_tr = make_time_features(S_tr).to(compute_device, non_blocking=True)
    T_va = make_time_features(S_va).to(compute_device, non_blocking=True)
    T_eval = make_time_features(S_eval).to(compute_device, non_blocking=True)

    class SimpleSTGNN(torch.nn.Module):
        def __init__(self, lookback, hidden=96, time_hidden=32):
            super().__init__()
            self.lookback = lookback

            self.hist_proj = torch.nn.Linear(lookback * 4, hidden)

            self.msg_proj1 = torch.nn.Linear(hidden, hidden, bias=False)
            self.msg_proj2 = torch.nn.Linear(hidden, hidden, bias=False)
            self.msg_proj3 = torch.nn.Linear(hidden, hidden, bias=False)

            self.scalar_proj = torch.nn.Sequential(
                torch.nn.Linear(22, hidden),
                torch.nn.GELU(),
                torch.nn.Linear(hidden, hidden),
            )

            self.time_mlp = torch.nn.Sequential(
                torch.nn.Linear(9, time_hidden),
                torch.nn.GELU(),
                torch.nn.Linear(time_hidden, time_hidden),
                torch.nn.GELU(),
            )

            self.node_norm = torch.nn.LayerNorm(hidden)

            self.decoder = torch.nn.Sequential(
                torch.nn.Linear(hidden + time_hidden + 22, hidden),
                torch.nn.GELU(),
                torch.nn.Linear(hidden, hidden // 2),
                torch.nn.GELU(),
                torch.nn.Linear(hidden // 2, 1),
            )

        def _graph_aggregate(self, x, A):
            return torch.einsum("ij,bjl->bil", A, x)

        def forward(self, x_hist, time_feat, A1, A2, A3):
            B, L, N = x_hist.shape

            x_nodes = x_hist.transpose(1, 2)  # [B, N, L]

            g1_hist = self._graph_aggregate(x_nodes, A1)
            g2_hist = self._graph_aggregate(x_nodes, A2)
            g3_hist = self._graph_aggregate(x_nodes, A3)

            hist_feat = torch.cat([x_nodes, g1_hist, g2_hist, g3_hist], dim=-1)
            hist = self.hist_proj(hist_feat)

            msg1 = self._graph_aggregate(hist, A1)
            msg2 = self._graph_aggregate(hist, A2)
            msg3 = self._graph_aggregate(hist, A3)
            h = hist + self.msg_proj1(msg1) + self.msg_proj2(msg2) + self.msg_proj3(msg3)
            h = F.gelu(h)

            first = x_hist[:, 0, :]
            last = x_hist[:, -1, :]
            mean_hist = x_hist.mean(dim=1)
            recent_k = x_hist[:, -min(3, L):, :].mean(dim=1)
            slope = (last - first) / max(self.lookback - 1, 1)

            if L >= 3:
                prev1 = x_hist[:, -2, :]
                prev2 = x_hist[:, -3, :]
                curvature = last - 2.0 * prev1 + prev2
            else:
                curvature = torch.zeros_like(last)

            neigh_last1 = last @ A1
            neigh_mean1 = mean_hist @ A1
            neigh_slope1 = slope @ A1

            neigh_last2 = last @ A2
            neigh_mean2 = mean_hist @ A2
            neigh_slope2 = slope @ A2

            neigh_last3 = last @ A3
            neigh_mean3 = mean_hist @ A3
            neigh_slope3 = slope @ A3

            degree_feat = torch.log1p(raw_deg.clamp_min(1e-6))[None, :].expand(B, -1)
            inv_degree_feat = torch.rsqrt(raw_deg.clamp_min(1e-6))[None, :].expand(B, -1)

            spatial_gap1 = last - neigh_last1
            spatial_gap2 = mean_hist - neigh_mean1
            spatial_gap3 = slope - neigh_slope1
            spatial_gap4 = neigh_last1 - neigh_last2
            spatial_gap5 = neigh_last2 - neigh_last3
            spatial_gap6 = torch.log1p(spatial_gap1.abs())

            scalar_feats = torch.stack(
                [
                    last,
                    recent_k,
                    mean_hist,
                    slope,
                    curvature,
                    neigh_last1,
                    neigh_mean1,
                    neigh_slope1,
                    neigh_last2,
                    neigh_mean2,
                    neigh_slope2,
                    neigh_last3,
                    neigh_mean3,
                    neigh_slope3,
                    degree_feat,
                    inv_degree_feat,
                    spatial_gap1,
                    spatial_gap2,
                    spatial_gap3,
                    spatial_gap4,
                    spatial_gap5,
                    spatial_gap6,
                ],
                dim=-1,
            )

            static = self.scalar_proj(scalar_feats)
            node_state = self.node_norm(F.gelu(h + static))

            time_emb = self.time_mlp(time_feat)

            step = time_feat[..., 6] * max(time_feat.shape[1] - 1, 1)
            base = (
                last[:, None, :]
                + slope[:, None, :] * step[:, :, None]
                + 0.5 * curvature[:, None, :] * (step[:, :, None] ** 2)
            )

            node_state = node_state[:, None, :, :].expand(-1, time_feat.shape[1], -1, -1)
            time_emb = time_emb[:, :, None, :].expand(-1, -1, node_state.shape[2], -1)
            stat_h = scalar_feats[:, None, :, :].expand(-1, time_feat.shape[1], -1, -1)

            dec_in = torch.cat([node_state, time_emb, stat_h], dim=-1)
            delta = self.decoder(dec_in).squeeze(-1)

            pred = base + delta

            spatial_smooth1 = pred @ A1
            spatial_smooth2 = pred @ A2
            pred = 0.76 * pred + 0.16 * spatial_smooth1 + 0.08 * spatial_smooth2
            return pred

    model = SimpleSTGNN(lookback=lookback).to(compute_device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

    best_state = None
    best_val = float("inf")
    patience = 8
    bad_epochs = 0
    epochs = 40
    epochs_run = 0

    for epoch in range(epochs):
        epochs_run = epoch + 1
        model.train()
        optimizer.zero_grad(set_to_none=True)

        pred_tr = model(X_tr, T_tr, A, A2, A3)
        loss = F.mse_loss(pred_tr, Y_tr)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_va = model(X_va, T_va, A, A2, A3)
            val_loss = F.mse_loss(pred_va, Y_va).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    if best_state is None:
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_eval_std = model(X_eval_std, T_eval, A, A2, A3)

    pred_eval = pred_eval_std * node_std[None, None, :] + node_mean[None, None, :]
    pred_eval = torch.expm1(pred_eval.clamp(min=-20.0, max=20.0))
    pred_eval = torch.nan_to_num(pred_eval, nan=0.0, posinf=0.0, neginf=0.0).clamp_min(0.0)

    return {
        "preds": pred_eval.detach().cpu(),
    }

##### ABLATION 1: NO SPATIAL

In [ ]:
import math
import torch
import torch.nn.functional as F


def _to_float_tensor(x):
    if torch.is_tensor(x):
        if getattr(x, "is_sparse", False) or getattr(x, "is_sparse_csr", False) or getattr(x, "is_sparse_bsr", False):
            x = x.to_dense()
        return x.float()
    if hasattr(x, "toarray"):
        x = x.toarray()
    elif hasattr(x, "A"):
        x = x.A
    return torch.tensor(x, dtype=torch.float32)


def _to_long_tensor(x):
    if torch.is_tensor(x):
        if getattr(x, "is_sparse", False) or getattr(x, "is_sparse_csr", False) or getattr(x, "is_sparse_bsr", False):
            x = x.to_dense()
        return x.long()
    if hasattr(x, "toarray"):
        x = x.toarray()
    elif hasattr(x, "A"):
        x = x.A
    return torch.tensor(x, dtype=torch.long)


def simple_tabular_st(
    train_input,
    train_target,
    x_eval=None,
    train_states=None,
    eval_states=None,
    graph=None,       # kept for compatibility; intentionally unused
    tid_sizes=None,
):
    x_eval = train_input if x_eval is None else x_eval
    eval_states = train_states if eval_states is None else eval_states
    tid_sizes = {"woy": 53} if tid_sizes is None else tid_sizes

    compute_device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    if not isinstance(compute_device, torch.device):
        compute_device = torch.device(compute_device)

    def _normalize_series_4d_to_3d(x):
        x = _to_float_tensor(x)
        if x.ndim == 4:
            if x.shape[-1] == 1:
                x = x[..., 0]
            else:
                x = x.mean(dim=-1)
        return x

    def _orient_series_x(x, expected_nodes=None):
        x = _normalize_series_4d_to_3d(x)

        if x.ndim == 1:
            x = x.unsqueeze(0).unsqueeze(-1)
        elif x.ndim == 2:
            if expected_nodes is not None and x.shape[1] == expected_nodes:
                x = x.unsqueeze(1)
            else:
                x = x.unsqueeze(-1)

        if x.ndim != 3:
            raise ValueError("train_input/x_eval must have shape [B, L, N], [B, N, L], or [B, L, N, C].")

        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        if expected_nodes is not None:
            if x.shape[-1] == expected_nodes:
                return x.contiguous()
            if x.shape[1] == expected_nodes:
                return x.transpose(1, 2).contiguous()

        return x.contiguous()

    def _orient_series_y(y, expected_nodes=None):
        y = _normalize_series_4d_to_3d(y)

        if y.ndim == 1:
            y = y.unsqueeze(0).unsqueeze(-1)
        elif y.ndim == 2:
            y = y.unsqueeze(-1)

        if y.ndim != 3:
            raise ValueError("train_target must have shape [B, H, N], [B, N, H], [B, H, N, C], or [B, H].")

        y = torch.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

        if expected_nodes is not None:
            if y.shape[-1] == expected_nodes:
                return y.contiguous()
            if y.shape[1] == expected_nodes:
                return y.transpose(1, 2).contiguous()

        return y.contiguous()

    if train_states is None or eval_states is None:
        raise ValueError("train_states and eval_states are required for temporal forecasting.")

    X_train_raw = _normalize_series_4d_to_3d(train_input)
    Y_train_raw = _normalize_series_4d_to_3d(train_target)

    if Y_train_raw.ndim == 2:
        Y_train_raw = Y_train_raw.unsqueeze(-1)

    if Y_train_raw.ndim != 3:
        raise ValueError("train_target must be 3D after preprocessing.")

    target_nodes = Y_train_raw.shape[-1] if Y_train_raw.shape[-1] > 1 else None

    X_train = _orient_series_x(X_train_raw, expected_nodes=target_nodes)
    Y_train = _orient_series_y(Y_train_raw, expected_nodes=X_train.shape[-1])
    X_eval = _orient_series_x(x_eval, expected_nodes=X_train.shape[-1])

    if X_train.ndim != 3 or X_eval.ndim != 3 or Y_train.ndim != 3:
        raise ValueError("All series inputs must be 3D after preprocessing.")

    if Y_train.shape[-1] != X_train.shape[-1]:
        if Y_train.shape[1] == X_train.shape[-1]:
            Y_train = Y_train.transpose(1, 2).contiguous()
        else:
            raise ValueError("train_target must match the node dimension of train_input.")

    n_nodes = X_train.shape[-1]

    if X_eval.shape[-1] != n_nodes and X_eval.shape[1] == n_nodes:
        X_eval = X_eval.transpose(1, 2).contiguous()

    X_train = torch.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    Y_train = torch.nan_to_num(Y_train, nan=0.0, posinf=0.0, neginf=0.0)
    X_eval = torch.nan_to_num(X_eval, nan=0.0, posinf=0.0, neginf=0.0)

    S_train = _to_long_tensor(train_states).to(compute_device, non_blocking=True)
    S_eval = _to_long_tensor(eval_states).to(compute_device, non_blocking=True)

    X_train = X_train.to(compute_device, non_blocking=True)
    Y_train = Y_train.to(compute_device, non_blocking=True)
    X_eval = X_eval.to(compute_device, non_blocking=True)

    B, lookback, n_nodes = X_train.shape
    horizon = Y_train.shape[1]
    lookback = max(int(lookback), 1)

    X_train = torch.log1p(X_train.clamp_min(0.0))
    Y_train = torch.log1p(Y_train.clamp_min(0.0))
    X_eval = torch.log1p(X_eval.clamp_min(0.0))

    if B <= 1:
        n_val = 0
    else:
        n_val = max(1, B // 5)
        n_val = min(n_val, B - 1)

    if n_val > 0:
        X_tr = X_train[:-n_val]
        Y_tr = Y_train[:-n_val]
        S_tr = S_train[:-n_val]

        X_va = X_train[-n_val:]
        Y_va = Y_train[-n_val:]
        S_va = S_train[-n_val:]
    else:
        X_tr = X_train
        Y_tr = Y_train
        S_tr = S_train

        X_va = X_train
        Y_va = Y_train
        S_va = S_train

    train_scale_source = torch.cat(
        [X_tr.reshape(-1, n_nodes), Y_tr.reshape(-1, n_nodes)],
        dim=0,
    )

    node_mean = train_scale_source.mean(dim=0)
    node_std = train_scale_source.std(dim=0, unbiased=False).clamp(min=1e-6)

    X_tr = (X_tr - node_mean[None, None, :]) / node_std[None, None, :]
    Y_tr = (Y_tr - node_mean[None, None, :]) / node_std[None, None, :]
    X_va = (X_va - node_mean[None, None, :]) / node_std[None, None, :]
    Y_va = (Y_va - node_mean[None, None, :]) / node_std[None, None, :]
    X_eval_std = (X_eval - node_mean[None, None, :]) / node_std[None, None, :]

    woy_size = max(int(tid_sizes.get("woy", 53)), 1)
    woy_period = float(woy_size)

    def make_time_features(states):
        if not torch.is_tensor(states):
            states = _to_long_tensor(states)

        states = states.to(compute_device)

        if states.ndim == 1:
            primary = states[:, None].repeat(1, horizon)
        elif states.ndim == 2:
            primary = states
        elif states.ndim >= 3:
            primary = states[..., 0]
        else:
            raise ValueError("states must have shape [B], [B, H], or [B, H, K].")

        if primary.shape[1] < horizon:
            pad = primary[:, -1:].repeat(1, horizon - primary.shape[1])
            primary = torch.cat([primary, pad], dim=1)
        elif primary.shape[1] > horizon:
            primary = primary[:, :horizon]

        primary = primary.float()

        woy = torch.remainder(primary, woy_period) / woy_period

        hpos = torch.linspace(
            0.0,
            1.0,
            horizon,
            device=states.device,
            dtype=torch.float32,
        )
        hpos = hpos.unsqueeze(0).expand(states.shape[0], -1)

        twopi = 2.0 * math.pi

        return torch.stack(
            [
                torch.sin(twopi * woy),
                torch.cos(twopi * woy),
                torch.sin(2.0 * twopi * woy),
                torch.cos(2.0 * twopi * woy),
                torch.sin(twopi * hpos),
                torch.cos(twopi * hpos),
                hpos,
                hpos ** 2,
                hpos ** 3,
            ],
            dim=-1,
        )

    T_tr = make_time_features(S_tr).to(compute_device, non_blocking=True)
    T_va = make_time_features(S_va).to(compute_device, non_blocking=True)
    T_eval = make_time_features(S_eval).to(compute_device, non_blocking=True)

    class SimpleTemporalAblation(torch.nn.Module):
        def __init__(self, lookback, hidden=96, time_hidden=32):
            super().__init__()
            self.lookback = lookback

            self.hist_proj = torch.nn.Linear(lookback, hidden)

            self.scalar_proj = torch.nn.Sequential(
                torch.nn.Linear(5, hidden),
                torch.nn.GELU(),
                torch.nn.Linear(hidden, hidden),
            )

            self.time_mlp = torch.nn.Sequential(
                torch.nn.Linear(9, time_hidden),
                torch.nn.GELU(),
                torch.nn.Linear(time_hidden, time_hidden),
                torch.nn.GELU(),
            )

            self.node_norm = torch.nn.LayerNorm(hidden)

            self.decoder = torch.nn.Sequential(
                torch.nn.Linear(hidden + time_hidden + 5, hidden),
                torch.nn.GELU(),
                torch.nn.Linear(hidden, hidden // 2),
                torch.nn.GELU(),
                torch.nn.Linear(hidden // 2, 1),
            )

        def forward(self, x_hist, time_feat):
            B, L, N = x_hist.shape

            x_nodes = x_hist.transpose(1, 2)  # [B, N, L]
            hist = self.hist_proj(x_nodes)

            first = x_hist[:, 0, :]
            last = x_hist[:, -1, :]
            mean_hist = x_hist.mean(dim=1)
            recent_k = x_hist[:, -min(3, L):, :].mean(dim=1)
            slope = (last - first) / max(self.lookback - 1, 1)

            if L >= 3:
                prev1 = x_hist[:, -2, :]
                prev2 = x_hist[:, -3, :]
                curvature = last - 2.0 * prev1 + prev2
            else:
                curvature = torch.zeros_like(last)

            scalar_feats = torch.stack(
                [
                    last,
                    recent_k,
                    mean_hist,
                    slope,
                    curvature,
                ],
                dim=-1,
            )

            static = self.scalar_proj(scalar_feats)
            node_state = self.node_norm(F.gelu(hist + static))

            time_emb = self.time_mlp(time_feat)

            step = time_feat[..., 6] * max(time_feat.shape[1] - 1, 1)

            base = (
                last[:, None, :]
                + slope[:, None, :] * step[:, :, None]
                + 0.5 * curvature[:, None, :] * (step[:, :, None] ** 2)
            )

            node_state = node_state[:, None, :, :].expand(
                -1,
                time_feat.shape[1],
                -1,
                -1,
            )

            time_emb = time_emb[:, :, None, :].expand(
                -1,
                -1,
                node_state.shape[2],
                -1,
            )

            stat_h = scalar_feats[:, None, :, :].expand(
                -1,
                time_feat.shape[1],
                -1,
                -1,
            )

            dec_in = torch.cat([node_state, time_emb, stat_h], dim=-1)
            delta = self.decoder(dec_in).squeeze(-1)

            pred = base + delta
            return pred

    model = SimpleTemporalAblation(lookback=lookback).to(compute_device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-3,
        weight_decay=1e-4,
    )

    best_state = None
    best_val = float("inf")
    patience = 8
    bad_epochs = 0
    epochs = 40

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        pred_tr = model(X_tr, T_tr)
        loss = F.mse_loss(pred_tr, Y_tr)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_va = model(X_va, T_va)
            val_loss = F.mse_loss(pred_va, Y_va).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {
                k: v.detach().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    if best_state is None:
        best_state = {
            k: v.detach().clone()
            for k, v in model.state_dict().items()
        }

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_eval_std = model(X_eval_std, T_eval)

    pred_eval = pred_eval_std * node_std[None, None, :] + node_mean[None, None, :]
    pred_eval = torch.expm1(pred_eval.clamp(min=-20.0, max=20.0))
    pred_eval = torch.nan_to_num(
        pred_eval,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).clamp_min(0.0)

    return {
        "preds": pred_eval.detach().cpu(),
    }

#### ILI2019

In [ ]:
# Cell 1: imports + seed

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sys

# sys.path.append("/net/dali/home/mscbio/alt245/epipatch")
from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# Cell 2: data loading + split construction

def build_splits(
    lookback=12,
    horizon=1,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/ILI2019.csv",
    adj_path="rawData/processed/ILI2019_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df.index = pd.to_datetime(data_df.index)
    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(data_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    woy = torch.tensor(data_df.index.isocalendar().week.values - 1, dtype=torch.long)
    dataset.states = torch.stack([woy], dim=-1)
    tid_s = {"woy": 53}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return data_df, dataset.graph, splits, tid_s


# Cell 4: evaluation helpers

def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]
    return eval_metrics(preds, targets)


# Cell 5: rolling retraining

def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    model_name="SimpleSTGNN",
    horizon=4,
    lookback=12,
    retrain_every=8,
    retrain_train_length=100,
    first_target=None,
    out_dir="outputs_retrain",
    device="cpu",
    dtw_matrix=None,
    epochs=50,
    use_future_ti=True,
    epi_mode=False,
    use_einn=False,
    loss_name="mse",
    tag_prefix="",
    retrain_schedule=None,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    woy = torch.tensor(data_df.index.isocalendar().week.values - 1, dtype=torch.long)
    states = torch.stack([woy], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true[local_i, 0, state_idx].item(),
                        "y_pred": y_pred[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


##### MAIN ######

horizon_baselines = {}

horizon_baselines[1] = {
    "mse": torch.as_tensor(0.3924),
    "mae": torch.as_tensor(0.3534),
    "rmse": torch.as_tensor(0.6265),
    "mse_filtered": torch.as_tensor(0.2315),
    "mae_filtered": torch.as_tensor(0.2979),
    "medse": torch.as_tensor(0.0305),
    "medae": torch.as_tensor(0.1747),
}

horizon_baselines[4] = {
    "mse": torch.as_tensor(1.7123),
    "mae": torch.as_tensor(0.7550),
    "rmse": torch.as_tensor(1.3085),
    "mse_filtered": torch.as_tensor(1.0226),
    "mae_filtered": torch.as_tensor(0.6254),
    "medse": torch.as_tensor(0.1277),
    "medae": torch.as_tensor(0.3573),
}

# Cell 6: config
dataset_name = "ILI2019"
lookback = 12
plot_state = "PA"
out_dir = f"simple_spatial_tid_regression_{dataset_name}"

horizon_val_metrics = []

for horizon in [1, 2, 4]:
    tag = f"simple_stgnn__horizon={horizon}"

    data_df, adj, splits, tid_s = build_splits(
        lookback=lookback,
        horizon=horizon,
        train_rate=0.6,
        val_rate=0.2,
        permute=False,
        data_path="rawData/processed/ILI2019.csv",
        adj_path="rawData/processed/ILI2019_adj.csv",
    )

    metrics_out = run_experiment(
        splits=splits,
        adj=adj,
        tid_s=tid_s,
    )

    results_df = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

    first_target = lookback + horizon - 1
    retrain_schedule = []
    for start_idx in range(first_target, len(data_df), 8):
        retrain_schedule.append(
            {
                "start_idx": start_idx,
                "train_start": max(0, start_idx - 100),
                "train_end": start_idx,
                "target_indices": list(range(start_idx, min(start_idx + 8, len(data_df)))),
            }
        )

    print("tuso_model_start")
    retrain_df = run_retraining(
        dataset_name=dataset_name,
        data_df=data_df,
        adj=adj,
        tid_s=tid_s,
        horizon=horizon,
        lookback=lookback,
        retrain_every=8,
        retrain_train_length=100,
        first_target=first_target,
        out_dir=out_dir,
        tag_prefix=tag,
        retrain_schedule=retrain_schedule,
    )
    print("tuso_model_end")

    metrics_all = eval_metrics(
        torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
        torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
    )
    horizon_val_metrics.append(metrics_all)
    
    #Save
    os.makedirs("optaris_epipatch/gnn", exist_ok=True)

    raw_pred_path = f"optaris_epipatch/gnn/ILI2019_tusoai_horizon={horizon}.csv"

    retrain_df_to_save = retrain_df.copy()
    retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
    retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
    retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
    retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
    retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

    retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
    retrain_df_to_save.to_csv(raw_pred_path)


#### JHUCASE

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=28,
    horizon=7,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/JHUcase.csv",
    adj_path="rawData/processed/JHUcase_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df.index = pd.to_datetime(data_df.index)

    # Match the preprocessing from the JHUcase script:
    # clip negatives, divide positive values by per-node std, keep zeros at zero.
    raw_df = data_df.astype(np.float32).copy()
    raw_df = raw_df.clip(lower=0.0)

    pos_mask = raw_df > 0
    pos_vals = raw_df.where(pos_mask)
    std_s = pos_vals.std(axis=0, skipna=True).replace(0, 1.0).fillna(1.0)

    scaler = {
        "std": torch.as_tensor(std_s.values, dtype=torch.float32),
        "zero_preserve": True,
        "center": False,
    }

    scaled_df = raw_df.copy()
    nonzero = scaled_df > 0
    scaled_df[nonzero] = scaled_df[nonzero] / std_s
    scaled_df[~nonzero] = 0.0

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    dow = torch.tensor(scaled_df.index.dayofweek.values, dtype=torch.long)
    dataset.states = torch.stack([dow], dim=-1)

    # simple_tabular_st expects tid_sizes["woy"] and only uses the first state channel.
    # We reuse that slot for a 7-day cyclical feature.
    tid_s = {"woy": 7}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=7,
    lookback=28,
    retrain_every=90,
    retrain_train_length=180,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    dow = torch.tensor(data_df.index.dayofweek.values, dtype=torch.long)
    states = torch.stack([dow], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # This matches your current procedure: for multi-step horizons,
        # only save the first forecast step [:, 0, :].
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "JHUcase"
    lookback = 28
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 4, 7, 10, 14, 28]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/JHUcase.csv",
            adj_path="rawData/processed/JHUcase_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 90):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 180),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 90, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,   # scaled data for training/inference
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=90,
            retrain_train_length=180,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,  # save outputs back on original scale
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()

#### NCHSDEATHS

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=12,
    horizon=4,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/NCHSdeaths.csv",
    adj_path="rawData/processed/NCHSdeaths_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df.index = pd.to_datetime(data_df.index)

    # Match the original NCHSdeaths preprocessing:
    # clip negatives, divide positive values by per-node std, keep zeros at zero.
    raw_df = data_df.astype(np.float32).copy()
    raw_df = raw_df.clip(lower=0.0)

    pos_mask = raw_df > 0
    pos_vals = raw_df.where(pos_mask)
    std_s = pos_vals.std(axis=0, skipna=True).replace(0, 1.0).fillna(1.0)

    scaler = {
        "std": torch.as_tensor(std_s.values, dtype=torch.float32),
        "zero_preserve": True,
        "center": False,
    }

    scaled_df = raw_df.copy()
    nonzero = scaled_df > 0
    scaled_df[nonzero] = scaled_df[nonzero] / std_s
    scaled_df[~nonzero] = 0.0

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    woy = torch.tensor(scaled_df.index.isocalendar().week.values - 1, dtype=torch.long)
    dataset.states = torch.stack([woy], dim=-1)
    tid_s = {"woy": 53}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=4,
    lookback=12,
    retrain_every=8,
    retrain_train_length=100,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    woy = torch.tensor(data_df.index.isocalendar().week.values - 1, dtype=torch.long)
    states = torch.stack([woy], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # Same behavior as your current setup:
        # for horizon > 1, only the first forecast step is saved.
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "NCHSdeaths"
    lookback = 12
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 2, 4, 8]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/NCHSdeaths.csv",
            adj_path="rawData/processed/NCHSdeaths_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 8):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 100),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 8, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,   # scaled data for training/inference
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=8,
            retrain_train_length=100,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,  # save back on original scale
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=28,
    horizon=7,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/HHShosp.csv",
    adj_path="rawData/processed/HHShosp_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df.index = pd.to_datetime(data_df.index)

    # Match the original HHShosp preprocessing:
    # clip negatives, divide positive values by per-node std, keep zeros at zero.
    raw_df = data_df.astype(np.float32).copy()
    raw_df = raw_df.clip(lower=0.0)

    pos_mask = raw_df > 0
    pos_vals = raw_df.where(pos_mask)
    std_s = pos_vals.std(axis=0, skipna=True).replace(0, 1.0).fillna(1.0)

    scaler = {
        "std": torch.as_tensor(std_s.values, dtype=torch.float32),
        "zero_preserve": True,
        "center": False,
    }

    scaled_df = raw_df.copy()
    nonzero = scaled_df > 0
    scaled_df[nonzero] = scaled_df[nonzero] / std_s
    scaled_df[~nonzero] = 0.0

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    # HHShosp uses day-of-week; simple_tabular_st expects tid_sizes["woy"],
    # so reuse that slot with period 7.
    dow = torch.tensor(scaled_df.index.dayofweek.values, dtype=torch.long)
    dataset.states = torch.stack([dow], dim=-1)
    tid_s = {"woy": 7}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=7,
    lookback=28,
    retrain_every=90,
    retrain_train_length=180,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    dow = torch.tensor(data_df.index.dayofweek.values, dtype=torch.long)
    states = torch.stack([dow], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # Same behavior as your current setup:
        # for horizon > 1, only the first forecast step is saved.
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "HHShosp"
    lookback = 28
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 4, 7, 10, 14, 28]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/HHShosp.csv",
            adj_path="rawData/processed/HHShosp_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 90):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 180),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 90, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,   # scaled data for training/inference
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=90,
            retrain_train_length=180,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=28,
    horizon=7,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/AUcase.csv",
    adj_path="rawData/processed/AUcase_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df.index = pd.to_datetime(data_df.index)

    # Match the original AUcase preprocessing:
    # clip negatives, divide positive values by per-node std, keep zeros at zero.
    raw_df = data_df.astype(np.float32).copy()
    raw_df = raw_df.clip(lower=0.0)

    pos_mask = raw_df > 0
    pos_vals = raw_df.where(pos_mask)
    std_s = pos_vals.std(axis=0, skipna=True).replace(0, 1.0).fillna(1.0)

    scaler = {
        "std": torch.as_tensor(std_s.values, dtype=torch.float32),
        "zero_preserve": True,
        "center": False,
    }

    scaled_df = raw_df.copy()
    nonzero = scaled_df > 0
    scaled_df[nonzero] = scaled_df[nonzero] / std_s
    scaled_df[~nonzero] = 0.0

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    # AUcase uses day-of-week; simple_tabular_st expects tid_sizes["woy"],
    # so reuse that slot with period 7.
    dow = torch.tensor(scaled_df.index.dayofweek.values, dtype=torch.long)
    dataset.states = torch.stack([dow], dim=-1)
    tid_s = {"woy": 7}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=7,
    lookback=28,
    retrain_every=90,
    retrain_train_length=180,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    dow = torch.tensor(data_df.index.dayofweek.values, dtype=torch.long)
    states = torch.stack([dow], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # Same behavior as your current setup:
        # for horizon > 1, only the first forecast step is saved.
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "AUcase"
    lookback = 28
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 4, 7, 10, 14, 28]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/AUcase.csv",
            adj_path="rawData/processed/AUcase_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 90):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 180),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 90, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,   # scaled data for training/inference
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=90,
            retrain_train_length=180,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=28,
    horizon=7,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/CAcase.csv",
    adj_path="rawData/processed/CAcase_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df.index = pd.to_datetime(data_df.index)

    # Match the original CAcase preprocessing:
    # clip negatives, divide positive values by per-node std, keep zeros at zero.
    raw_df = data_df.astype(np.float32).copy()
    raw_df = raw_df.clip(lower=0.0)

    pos_mask = raw_df > 0
    pos_vals = raw_df.where(pos_mask)
    std_s = pos_vals.std(axis=0, skipna=True).replace(0, 1.0).fillna(1.0)

    scaler = {
        "std": torch.as_tensor(std_s.values, dtype=torch.float32),
        "zero_preserve": True,
        "center": False,
    }

    scaled_df = raw_df.copy()
    nonzero = scaled_df > 0
    scaled_df[nonzero] = scaled_df[nonzero] / std_s
    scaled_df[~nonzero] = 0.0

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    # CAcase uses day-of-week; simple_tabular_st expects tid_sizes["woy"],
    # so reuse that slot with period 7.
    dow = torch.tensor(scaled_df.index.dayofweek.values, dtype=torch.long)
    dataset.states = torch.stack([dow], dim=-1)
    tid_s = {"woy": 7}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=7,
    lookback=28,
    retrain_every=90,
    retrain_train_length=180,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    dow = torch.tensor(data_df.index.dayofweek.values, dtype=torch.long)
    states = torch.stack([dow], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # Same behavior as your current setup:
        # for horizon > 1, only the first forecast step is saved.
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "CAcase"
    lookback = 28
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 4, 7, 10, 14, 28]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/CAcase.csv",
            adj_path="rawData/processed/CAcase_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 90):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 180),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 90, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,   # scaled data for training/inference
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=90,
            retrain_train_length=180,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()

### CANPOSITIVITY

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=28,
    horizon=7,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/CANpositivity.csv",
    adj_path="rawData/processed/CANpositivity_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df = data_df.fillna(0.0).astype(np.float32)
    data_df.index = pd.to_datetime(data_df.index)

    # No scaling here; keep behavior aligned with the second file's setup.
    raw_df = data_df.copy()
    scaled_df = data_df.copy()
    scaler = None

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    # The second file uses day-of-week states.
    # Keep the original script interface for simple_tabular_st by reusing
    # the expected tid_sizes slot name.
    dow = torch.tensor(scaled_df.index.dayofweek.values, dtype=torch.long)
    dataset.states = torch.stack([dow], dim=-1)
    tid_s = {"woy": 7}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=7,
    lookback=28,
    retrain_every=90,
    retrain_train_length=180,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    dow = torch.tensor(data_df.index.dayofweek.values, dtype=torch.long)
    states = torch.stack([dow], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # Same behavior as the original script:
        # for horizon > 1, only the first forecast step is saved.
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "CANpositivity"
    lookback = 28
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 4, 7, 10, 14, 28]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/CANpositivity.csv",
            adj_path="rawData/processed/CANpositivity_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 90):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 180),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 90, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=90,
            retrain_train_length=180,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()

### DVCLI

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=28,
    horizon=7,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/DVcli.csv",
    adj_path="rawData/processed/DVcli_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df = data_df.fillna(0.0).astype(np.float32)
    data_df = data_df / 100.0
    data_df.index = pd.to_datetime(data_df.index)

    # No extra scaling beyond the dataset's own preprocessing.
    raw_df = data_df.copy()
    scaled_df = data_df.copy()
    scaler = None

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    # DVcli config uses day-of-week states.
    # Keep the original simple_tabular_st interface by reusing the slot name
    # expected by the original script.
    dow = torch.tensor(scaled_df.index.dayofweek.values, dtype=torch.long)
    dataset.states = torch.stack([dow], dim=-1)
    tid_s = {"woy": 7}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=7,
    lookback=28,
    retrain_every=90,
    retrain_train_length=180,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    dow = torch.tensor(data_df.index.dayofweek.values, dtype=torch.long)
    states = torch.stack([dow], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # Same behavior as the original script:
        # for horizon > 1, only the first forecast step is saved.
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "DVcli"
    lookback = 28
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 4, 7, 10, 14, 28]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/DVcli.csv",
            adj_path="rawData/processed/DVcli_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 90):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 180),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 90, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=90,
            retrain_train_length=180,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()

### CHNGINPATIENT

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=28,
    horizon=7,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/CHNGinpatient.csv",
    adj_path="rawData/processed/CHNGinpatient_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df = data_df.fillna(0.0).astype(np.float32)
    data_df.index = pd.to_datetime(data_df.index)

    # No extra scaling beyond the dataset's own preprocessing.
    raw_df = data_df.copy()
    scaled_df = data_df.copy()
    scaler = None

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    # CHNGinpatient config uses day-of-week states.
    # Keep the original simple_tabular_st interface by reusing the slot name
    # expected by the original script.
    dow = torch.tensor(scaled_df.index.dayofweek.values, dtype=torch.long)
    dataset.states = torch.stack([dow], dim=-1)
    tid_s = {"woy": 7}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=7,
    lookback=28,
    retrain_every=90,
    retrain_train_length=180,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    dow = torch.tensor(data_df.index.dayofweek.values, dtype=torch.long)
    states = torch.stack([dow], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # Same behavior as the original script:
        # for horizon > 1, only the first forecast step is saved.
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "CHNGinpatient"
    lookback = 28
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 4, 7, 10, 14, 28]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/CHNGinpatient.csv",
            adj_path="rawData/processed/CHNGinpatient_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 90):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 180),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 90, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=90,
            retrain_train_length=180,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()

### CHNGOUTPATIENT

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=28,
    horizon=7,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/CHNGoutpatient.csv",
    adj_path="rawData/processed/CHNGoutpatient_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df = data_df.fillna(0.0).astype(np.float32)
    data_df.index = pd.to_datetime(data_df.index)

    # No extra scaling beyond the dataset's own preprocessing.
    raw_df = data_df.copy()
    scaled_df = data_df.copy()
    scaler = None

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    # CHNGoutpatient config uses day-of-week states.
    # Keep the original simple_tabular_st interface by reusing the slot name
    # expected by the original script.
    dow = torch.tensor(scaled_df.index.dayofweek.values, dtype=torch.long)
    dataset.states = torch.stack([dow], dim=-1)
    tid_s = {"woy": 7}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=7,
    lookback=28,
    retrain_every=90,
    retrain_train_length=180,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    dow = torch.tensor(data_df.index.dayofweek.values, dtype=torch.long)
    states = torch.stack([dow], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # Same behavior as the original script:
        # for horizon > 1, only the first forecast step is saved.
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "CHNGoutpatient"
    lookback = 28
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 4, 7, 10, 14, 28]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/CHNGoutpatient.csv",
            adj_path="rawData/processed/CHNGoutpatient_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 90):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 180),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 90, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=90,
            retrain_train_length=180,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()

### CPRADMISSIONS

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

# Assumes simple_tabular_st(...) is already defined above this script
# or imported from another file.


def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def build_splits(
    lookback=28,
    horizon=7,
    train_rate=0.6,
    val_rate=0.2,
    permute=False,
    data_path="rawData/processed/CPRadmissions.csv",
    adj_path="rawData/processed/CPRadmissions_adj.csv",
):
    data_df = pd.read_csv(data_path, index_col=0)
    data_df.index = pd.to_datetime(data_df.index)

    # Match the CPRadmissions preprocessing:
    # clip negatives, divide positive values by per-node std, keep zeros at zero.
    raw_df = data_df.astype(np.float32).copy()
    raw_df = raw_df.clip(lower=0.0)

    pos_mask = raw_df > 0
    pos_vals = raw_df.where(pos_mask)
    std_s = pos_vals.std(axis=0, skipna=True).replace(0, 1.0).fillna(1.0)

    scaler = {
        "std": torch.as_tensor(std_s.values, dtype=torch.float32),
        "zero_preserve": True,
        "center": False,
    }

    scaled_df = raw_df.copy()
    nonzero = scaled_df > 0
    scaled_df[nonzero] = scaled_df[nonzero] / std_s
    scaled_df[~nonzero] = 0.0

    adj_df = pd.read_csv(adj_path, index_col=0)

    dataset = UniversalDataset()
    data = np.expand_dims(scaled_df.values, axis=-1)

    dataset.x = torch.tensor(data, dtype=torch.float32)
    dataset.y = dataset.x[:, :, 0]
    dataset.graph = torch.tensor(adj_df.to_numpy(), dtype=torch.float32)

    # CPRadmissions config uses day-of-week states.
    # Keep the original simple_tabular_st interface by reusing the slot name
    # expected by the original script.
    dow = torch.tensor(scaled_df.index.dayofweek.values, dtype=torch.long)
    dataset.states = torch.stack([dow], dim=-1)
    tid_s = {"woy": 7}

    train_dataset, val_dataset, test_dataset = dataset.ganerate_splits(
        train_rate=train_rate,
        val_rate=val_rate,
    )

    train_input, train_target, _, train_states_future, train_adj = generate_dataset(
        X=train_dataset["features"],
        Y=train_dataset["target"],
        states=train_dataset["states"],
        dynamic_adj=train_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    val_input, val_target, _, val_states_future, val_adj = generate_dataset(
        X=val_dataset["features"],
        Y=val_dataset["target"],
        states=val_dataset["states"],
        dynamic_adj=val_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )
    test_input, test_target, _, test_states_future, test_adj = generate_dataset(
        X=test_dataset["features"],
        Y=test_dataset["target"],
        states=test_dataset["states"],
        dynamic_adj=test_dataset["dynamic_graph"],
        lookback_window_size=lookback,
        horizon=horizon,
        permute=permute,
    )

    splits = {
        "train": {
            "features": train_input,
            "targets": train_target,
            "states": train_states_future,
            "dynamic_graph": train_adj,
        },
        "val": {
            "features": val_input,
            "targets": val_target,
            "states": val_states_future,
            "dynamic_graph": val_adj,
        },
        "test": {
            "features": test_input,
            "targets": test_target,
            "states": test_states_future,
            "dynamic_graph": test_adj,
        },
    }

    return raw_df, scaled_df, dataset.graph, splits, tid_s, scaler


def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)

    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "medse": medse,
        "medae": medae,
    }


def save_metrics(metrics_out, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)

    row = {}
    for k, v in metrics_out.items():
        if torch.is_tensor(v):
            row[k] = v.item()
        elif isinstance(v, (np.floating, np.integer)):
            row[k] = float(v)
        else:
            row[k] = v

    row["tag"] = tag

    path = os.path.join(out_dir, f"{tag}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, default=float)

    return row


def run_experiment(splits, adj, tid_s, scaler=None, inverse_scale_for_metrics=True):
    out_model = simple_tabular_st(
        train_input=splits["train"]["features"],
        train_target=splits["train"]["targets"],
        x_eval=splits["test"]["features"],
        train_states=splits["train"]["states"],
        eval_states=splits["test"]["states"],
        graph=adj,
        tid_sizes=tid_s,
    )

    preds = out_model["preds"]
    targets = splits["test"]["targets"]

    if inverse_scale_for_metrics and scaler is not None:
        std = scaler["std"].to(preds.device, dtype=preds.dtype)
        preds = preds * std[None, None, :]
        targets = targets.to(preds.device, dtype=preds.dtype) * std[None, None, :]

    return eval_metrics(preds, targets)


def run_retraining(
    dataset_name,
    data_df,
    adj,
    tid_s,
    scaler,
    horizon=7,
    lookback=28,
    retrain_every=90,
    retrain_train_length=180,
    first_target=None,
    out_dir="outputs_retrain",
    tag_prefix="",
    retrain_schedule=None,
    inverse_scale_for_save=True,
):
    n_total = len(data_df)

    if first_target is None:
        first_target = lookback + horizon - 1

    values = torch.tensor(np.expand_dims(data_df.values, axis=-1), dtype=torch.float32)
    targets_full = values[:, :, 0]

    dow = torch.tensor(data_df.index.dayofweek.values, dtype=torch.long)
    states = torch.stack([dow], dim=-1)

    full_X, full_y, _, full_states, _ = generate_dataset(
        X=values,
        Y=targets_full,
        states=states,
        dynamic_adj=None,
        lookback_window_size=lookback,
        horizon=horizon,
        permute=False,
    )

    if retrain_schedule is None:
        retrain_schedule = []
        for start_idx in range(first_target, n_total, retrain_every):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - retrain_train_length),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + retrain_every, n_total))),
                }
            )

    rows = []
    retrain_id = 0
    std_vec = scaler["std"].to(torch.float32) if scaler is not None else None

    for schedule in retrain_schedule:
        train_start = schedule["train_start"]
        train_end = schedule["train_end"]
        target_indices = schedule["target_indices"]

        subset_values = values[train_start:train_end]
        subset_targets = targets_full[train_start:train_end]
        subset_states = states[train_start:train_end]

        train_input, train_target, _, train_states_future, _ = generate_dataset(
            X=subset_values,
            Y=subset_targets,
            states=subset_states,
            dynamic_adj=None,
            lookback_window_size=lookback,
            horizon=horizon,
            permute=False,
        )

        if train_input.numel() == 0:
            continue

        offset = lookback + horizon - 1
        sample_ids = [t - offset for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]
        valid_targets = [t for t in target_indices if 0 <= (t - offset) < full_X.shape[0]]

        if len(sample_ids) == 0:
            continue

        x_eval = full_X[sample_ids]
        y_true = full_y[sample_ids]
        states_eval = full_states[sample_ids]

        y_pred = simple_tabular_st(
            train_input=train_input,
            train_target=train_target,
            x_eval=x_eval,
            train_states=train_states_future,
            eval_states=states_eval,
            graph=adj,
            tid_sizes=tid_s,
        )["preds"]

        y_pred = y_pred.reshape(y_true.shape)

        if inverse_scale_for_save and std_vec is not None:
            y_true_save = y_true * std_vec[None, None, :]
            y_pred_save = y_pred * std_vec[None, None, :]
        else:
            y_true_save = y_true
            y_pred_save = y_pred

        # Same behavior as the original script:
        # for horizon > 1, only the first forecast step is saved.
        for local_i, t_idx in enumerate(valid_targets):
            for state_idx, state_name in enumerate(data_df.columns):
                rows.append(
                    {
                        "retrain_id": retrain_id,
                        "timestamp": data_df.index[t_idx],
                        "state_idx": state_idx,
                        "state": str(state_name),
                        "train_start": data_df.index[train_start],
                        "train_end": data_df.index[train_end - 1],
                        "eval_start": data_df.index[target_indices[0]],
                        "eval_end": data_df.index[target_indices[-1]],
                        "y_true": y_true_save[local_i, 0, state_idx].item(),
                        "y_pred": y_pred_save[local_i, 0, state_idx].item(),
                    }
                )

        retrain_id += 1

    retrain_df = pd.DataFrame(rows)
    if retrain_df.empty:
        return retrain_df

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"retrain_{dataset_name}_{tag_prefix}.csv")
    retrain_df.to_csv(csv_path, index=False)
    return retrain_df


def main():
    dataset_name = "CPRadmissions"
    lookback = 28
    raw_save_dir = "optaris_epipatch/gnn"

    os.makedirs(raw_save_dir, exist_ok=True)

    horizon_val_metrics = []

    for horizon in [1, 4, 7, 10, 14, 28]:
        tag = f"simple_stgnn__horizon={horizon}"

        raw_df, data_df, adj, splits, tid_s, scaler = build_splits(
            lookback=lookback,
            horizon=horizon,
            train_rate=0.6,
            val_rate=0.2,
            permute=False,
            data_path="rawData/processed/CPRadmissions.csv",
            adj_path="rawData/processed/CPRadmissions_adj.csv",
        )

        metrics_out = run_experiment(
            splits=splits,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            inverse_scale_for_metrics=True,
        )

        _ = pd.DataFrame([save_metrics(metrics_out, out_dir, tag)])

        first_target = lookback + horizon - 1
        retrain_schedule = []
        for start_idx in range(first_target, len(data_df), 90):
            retrain_schedule.append(
                {
                    "start_idx": start_idx,
                    "train_start": max(0, start_idx - 180),
                    "train_end": start_idx,
                    "target_indices": list(range(start_idx, min(start_idx + 90, len(data_df)))),
                }
            )

        print(f"tuso_model_start horizon={horizon}")
        retrain_df = run_retraining(
            dataset_name=dataset_name,
            data_df=data_df,
            adj=adj,
            tid_s=tid_s,
            scaler=scaler,
            horizon=horizon,
            lookback=lookback,
            retrain_every=90,
            retrain_train_length=180,
            first_target=first_target,
            out_dir=out_dir,
            tag_prefix=tag,
            retrain_schedule=retrain_schedule,
            inverse_scale_for_save=False,
        )
        print(f"tuso_model_end horizon={horizon}")

        if retrain_df.empty:
            print(f"No retraining rows produced for horizon={horizon}")
            continue

        metrics_all = eval_metrics(
            torch.tensor(retrain_df["y_pred"].to_numpy(), dtype=torch.float32),
            torch.tensor(retrain_df["y_true"].to_numpy(), dtype=torch.float32),
        )

        metrics_row = {"horizon": horizon}
        for k, v in metrics_all.items():
            if torch.is_tensor(v):
                metrics_row[k] = v.item()
            elif isinstance(v, (np.floating, np.integer)):
                metrics_row[k] = float(v)
            else:
                metrics_row[k] = v
        horizon_val_metrics.append(metrics_row)

        raw_pred_path = os.path.join(
            raw_save_dir,
            f"{dataset_name}_tusoai_horizon={horizon}.csv",
        )

        retrain_df_to_save = retrain_df.copy()
        retrain_df_to_save["timestamp"] = pd.to_datetime(retrain_df_to_save["timestamp"])
        retrain_df_to_save["train_start"] = pd.to_datetime(retrain_df_to_save["train_start"])
        retrain_df_to_save["train_end"] = pd.to_datetime(retrain_df_to_save["train_end"])
        retrain_df_to_save["eval_start"] = pd.to_datetime(retrain_df_to_save["eval_start"])
        retrain_df_to_save["eval_end"] = pd.to_datetime(retrain_df_to_save["eval_end"])

        retrain_df_to_save = retrain_df_to_save.set_index("retrain_id")
        retrain_df_to_save.to_csv(raw_pred_path)

        print(f"Saved raw predictions to: {raw_pred_path}")

    if horizon_val_metrics:
        metrics_df = pd.DataFrame(horizon_val_metrics)
        metrics_df.to_csv(
            os.path.join(raw_save_dir, f"{dataset_name}_tusoai_horizon_metrics.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()